# Stage 6 -- Leakage-Controlled T-side A/B Baseline Model

Out-of-fold baseline comparison for high-confidence Vitality T-side Mirage plants.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data/gold/modeling/t_side_ab_baseline'

def load(name):
    return pd.read_parquet(DATA / f'{name}.parquet')

audit = load('ab_model_dataset_audit')
features = load('ab_model_feature_sets')
metrics = load('ab_model_metrics')
confusion = load('ab_model_confusion_matrices')
predictions = load('ab_model_predictions')
importance = load('ab_model_feature_importance')
comparison = load('ab_model_horizon_comparison')

## Dataset audit and A/B balance

In [ ]:
display(audit.T.rename(columns={0: 'value'}))
balance = audit.iloc[0][['class_A', 'class_B']]
balance.index = ['A', 'B']
balance.plot(kind='bar', title='High-confidence planted labels')
plt.ylabel('Rounds')
plt.show()

## Metrics and majority comparison

In [ ]:
display(metrics[['horizon_seconds', 'model_name', 'balanced_accuracy', 'macro_f1', 'f1_A', 'f1_B', 'roc_auc']])
pivot = metrics.pivot(index='horizon_seconds', columns='model_name', values='macro_f1')
pivot.plot(marker='o', title='Out-of-fold macro F1 by horizon')
plt.ylabel('Macro F1')
plt.show()

## Confusion matrices

In [ ]:
for (horizon, model), group in confusion.groupby(['horizon_seconds', 'model_name']):
    matrix = group.pivot(index='true_label', columns='predicted_label', values='count')
    print(f'{horizon}s -- {model}')
    display(matrix)

## Feature importance

In [ ]:
top_importance = (importance.assign(abs_importance=importance['importance_value'].abs())
                  .sort_values(['horizon', 'model_name', 'abs_importance'], ascending=[True, True, False])
                  .groupby(['horizon', 'model_name']).head(10))
display(top_importance[['horizon', 'model_name', 'feature_name', 'importance_value', 'direction']])

## Out-of-fold errors

In [ ]:
error_columns = ['horizon_seconds', 'model_name', 'round_feature_id', 'opponent', 'round_num', 'true_label', 'predicted_label', 'predicted_proba_B', 'fold_id']
display(predictions.loc[~predictions['is_correct'], error_columns].head(30))

## Horizon comparison

In [ ]:
display(features[['horizon_seconds', 'model_rows', 'rows_excluded_plant_before_horizon', 'total_selected_features']])
display(comparison)

## Next

Next: refine features/model after manual review